# 1.2 The bias-complexity tradeoff

Total error splits in two: how good the best model in your class could be, and how well finite data lets you find it.

Richer classes reduce approximation error but raise estimation error. The right model class is therefore a capacity decision conditioned on sample size, not a one-way race toward expressiveness.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)


def estimation_error(complexity, m):
    return np.sqrt(complexity / (2 * m))


def total_error(approx, complexity, m):
    return approx + estimation_error(complexity, m)


def make_class_table(approxs, complexities, m, name):
    rows = []
    for idx, approx in enumerate(approxs):
        complexity = complexities[idx]
        total = total_error(approx, complexity, m)
        rows.append({
            "rung": idx + 1,
            "name": name,
            "approx": float(approx),
            "complexity": float(complexity),
            "m": int(m),
            "estimation": float(estimation_error(complexity, m)),
            "total": float(total),
        })
    return rows


def select_min_total(rows):
    totals = [row["total"] for row in rows]
    index = int(np.argmin(totals))
    return rows[index]


def noisy_floor(approx, floor):
    return np.maximum(np.asarray(approx, dtype=float), floor)

## The concept, built once (D1)

The lesson uses the excess-risk proxy

$$	ext{total}(H,m)=	ext{approx}(H)+\sqrt{rac{	ext{complexity}(H)}{2m}}.$$

At $m=100$, the approximation values are $0.20,0.08,0.02$ and the complexity values are $10,100,1000$.

In [ ]:
lesson_approx = np.array([0.20, 0.08, 0.02])
lesson_complexity = np.array([10, 100, 1000])
lesson_rows = make_class_table(lesson_approx, lesson_complexity, 100, "D1 lesson classes")
for row in lesson_rows:
    print(row)

The expected lesson totals are $0.424$, $0.787$, and $2.256$ after rounding. The medium-class estimation term at $m=1600$ should be $\sqrt{100/(2\cdot1600)}=0.177$.

In [ ]:
lesson_totals = np.array([row["total"] for row in lesson_rows])
assert np.allclose(np.round(lesson_totals, 3), [0.424, 0.787, 2.256])
medium_est_1600 = estimation_error(100, 1600)
assert np.isclose(np.round(medium_est_1600, 3), 0.177)
print("D1 totals:", np.round(lesson_totals, 3))
print("medium estimation at m=1600:", round(medium_est_1600, 3))

## The dataset ladder

D1 is the lesson's three-class hand table. D2 raises the class-capacity resolution, D3 holds the classes fixed while sweeping sample size, D4 adds an irreducible noisy approximation floor, and D5 offers an overlarge menu at small $m$.

In [ ]:
rungs = []
rungs.append(make_class_table(lesson_approx, lesson_complexity, 100, "D1 lesson"))
complexity_d2 = np.array([5, 10, 30, 100, 300, 1000, 3000])
approx_d2 = 0.015 + 0.32 / np.sqrt(complexity_d2)
rungs.append(make_class_table(approx_d2, complexity_d2, 100, "D2 more classes"))
rungs.append(make_class_table(lesson_approx, lesson_complexity, 1600, "D3 larger sample"))
approx_d4 = noisy_floor(approx_d2, 0.06)
rungs.append(make_class_table(approx_d4, complexity_d2, 400, "D4 noisy floor"))
complexity_d5 = np.array([10, 30, 100, 300, 1000, 3000, 10000, 30000])
approx_d5 = 0.005 + 0.25 / np.sqrt(complexity_d5)
rungs.append(make_class_table(approx_d5, complexity_d5, 50, "D5 overlarge menu"))

for rows in rungs:
    best = select_min_total(rows)
    print(rows[0]["name"])
    print("  classes=", len(rows))
    print("  m=", rows[0]["m"])
    print("  first row=", rows[0])
    print("  selected rung=", best["rung"])
    print("  selected total=", round(best["total"], 3))

In [ ]:
summary = []
for rows in rungs:
    best = select_min_total(rows)
    summary.append((rows[0]["name"], best["complexity"], best["total"]))
print("rung | selected complexity | minimum total")
for name, complexity, total in summary:
    print(f"{name}: {complexity:.0f} | {total:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))
for ax, rows in zip(axes[0], rungs):
    complexity = [row["complexity"] for row in rows]
    approx = [row["approx"] for row in rows]
    est = [row["estimation"] for row in rows]
    total = [row["total"] for row in rows]
    ax.plot(complexity, approx, marker="o", label="approx")
    ax.plot(complexity, est, marker="o", label="estimation")
    ax.plot(complexity, total, marker="o", label="total")
    ax.set_xscale("log")
    ax.set_title(rows[0]["name"])
    ax.grid(True, alpha=0.3)
axes[0, 0].legend()
best_complexities = [item[1] for item in summary]
best_totals = [item[2] for item in summary]
axes[1, 0].plot(best_complexities, best_totals, marker="o")
axes[1, 0].set_xscale("log")
axes[1, 0].set_xlabel("selected capacity")
axes[1, 0].set_ylabel("minimum total error")
axes[1, 0].set_title("Payoff curve: total error vs capacity")
axes[1, 0].grid(True, alpha=0.3)
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()

## Pitfall on the hardest rung

Pitfall: bigger model is always better. D5 makes the minimum approximation class enormous, but the finite-sample estimation term makes that choice lose badly.

In [ ]:
d5_rows = rungs[-1]
wrong = min(d5_rows, key=lambda row: row["approx"])
fixed = select_min_total(d5_rows)
print(f"wrong minimum-approx rung={wrong['rung']} complexity={wrong['complexity']:.0f} total={wrong['total']:.3f}")
print(f"fixed minimum-total rung={fixed['rung']} complexity={fixed['complexity']:.0f} total={fixed['total']:.3f}")
print(f"improvement={wrong['total'] - fixed['total']:.3f}")

## Evaluate it + Practice

- Metric: total excess-error proxy.
- No-skill baseline: choose the lowest approximation error only.
- Cheap sanity check: verify each estimation term shrinks by one half when m quadruples.
- Ablation: set estimation to zero and watch D5 overselect capacity.
- Failure signals: selected capacity at the edge of the menu or totals above one.

### Practice prompts

- Change m from 100 to 400 in D1 and predict the winning class.

- Raise the noisy floor in D4 and inspect when capacity stops helping.

- Add a class with complexity 1,000,000 and compute its total.